# Dataset Exploration before sentiment analysis

Describe the analytical corpus **before** estimating sentiment with FinBERT or Financial-RoBERTa. The analyses examine coverage, text composition, Leadership Context (PERF, STRAT, MARKET, CHALL, PEOPLE, STAKE, OTHER), manually coded Information Valence (positive, neutral, negative), and practical issues that could influence subsequent models.

**Sample:** Adidas and Puma (Germany), Asics and Mizuno (Japan), 2020–2025. Mizuno 2020 is expected to be absent because the thesis excludes it for lack of an official English version.

- Publication-quality PNG and vector PDF figures in `processed/figures/`
- CSV tables in `processed/tables/`
- diagnostics in `processed/quality_reports/`


## 1. Environment, paths and reproducibility

In [1]:
from pathlib import Path
import re
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from IPython.display import display


PROJECT_ROOT = None  
INPUT_NAMES = ("adidas", "puma", "asics", "mizuno")
ANALYTICAL_TEXT_TYPES = {"body_text", "answer", "quote"} # question and title are excluded by default because the unit of interest is leadership communication.
CONTEXT_ORDER = ["PERF", "STRAT", "MARKET", "CHALL", "PEOPLE", "STAKE", "OTHER"]
VALENCE_ORDER = ["negative", "neutral", "positive"]
COMPANY_ORDER = ["adidas", "puma", "asics", "mizuno"]
COUNTRY_ORDER = ["DEU", "JPN"]
YEARS = list(range(2020, 2026))
COMPANY_COUNTRY = {"adidas": "DEU", "puma": "DEU", "ssics": "JPN", "mizuno": "JPN"}

if PROJECT_ROOT is None:
    candidates = [Path.cwd(), *Path.cwd().parents]
    candidates += [Path.cwd() / "..", Path.cwd() / "project"]
    matches = [p.resolve() for p in candidates if (p / "data" / "input").is_dir()]
    if not matches:
        raise FileNotFoundError(
            "Could not find 'data/input' relative to the notebook working directory. "
            "Set PROJECT_ROOT to your thesis project folder. The actual XLSX data files "
            "must also be present to run this notebook."
        )
    PROJECT_ROOT = matches[0]
else:
    PROJECT_ROOT = Path(PROJECT_ROOT).expanduser().resolve()

DATA_DIR = PROJECT_ROOT / "data" / "input"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROCESSED_DIR / "results"
FIGURES_DIR = PROCESSED_DIR / "figures"
TABLES_DIR = PROCESSED_DIR / "tables"
QUALITY_DIR = PROCESSED_DIR / "quality_reports"
for folder in (FIGURES_DIR, TABLES_DIR, QUALITY_DIR):
    folder.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "font.size": 10,
})

# Figure registry makes it easier to reproduce the manuscript figure list.
figure_records = []
def save_figure(fig, filename, caption):
    """Export editable/vector-friendly PDF and high-resolution PNG."""
    stem = re.sub(r"[^a-z0-9_]+", "_", filename.lower()).strip("_")
    for ext in ("png", "pdf"):
        fig.savefig(FIGURES_DIR / f"{stem}.{ext}", dpi=300, bbox_inches="tight")
    figure_records.append({"figure": stem, "caption": caption,
                           "png": f"results/figures/{stem}.png",
                           "pdf": f"results/figures/{stem}.pdf"})
    plt.show()
    plt.close(fig)

def save_table(df, name, index=False):
    target = TABLES_DIR / f"{name}.csv"
    df.to_csv(target, index=index, encoding="utf-8-sig")
    return target

print("Project root:", PROJECT_ROOT)
print("Figures:", FIGURES_DIR)
print("Tables:", TABLES_DIR)

Project root: /Users/Saowalak/Desktop/CEOCommunication
Figures: /Users/Saowalak/Desktop/CEOCommunication/data/processed/figures
Tables: /Users/Saowalak/Desktop/CEOCommunication/data/processed/tables


## 2. Load the four company workbooks once

All company files are expected at `data/input/{company}.xlsx`. The first worksheet is read, as in the original validation step. No new manually assigned labels are created: the notebook uses the final `Leadership_context_human` and `Information_valence_human` fields. The raw file metadata are retained for auditing.

In [ ]:
required_columns = [
    "ID", "Text", "Text_type", "Leadership_context_human", "Information_valence_human",
    "Communication_type", "Source_type", "Company", "Country", "Year", "Covid",
    "Communicator_role", "Communicator_name", "Comment", "Coding_ambiguity", "Alternative_context"
]
frames = []
for basename in INPUT_NAMES:
    filepath = DATA_DIR / f"{basename}.xlsx"
    if not filepath.exists():
        raise FileNotFoundError(f"Missing input workbook: {filepath}")
    part = pd.read_excel(filepath, sheet_name=0, dtype={"ID": "string"})
    part.columns = part.columns.astype(str).str.strip()
    missing = [c for c in required_columns if c not in part.columns]
    if missing:
        raise ValueError(f"{filepath.name} is missing required columns: {missing}")
    part["_file_company"] = basename.capitalize()
    part["_file_name"] = filepath.name
    frames.append(part)

raw = pd.concat(frames, ignore_index=True, sort=False)
raw["ID"] = raw["ID"].astype("string").str.strip()
raw["Company"] = raw["Company"].astype("string").str.strip().str.title()
raw["Country_original"] = raw["Country"].copy()
raw["Country"] = raw["Country"].astype("string").str.strip().str.lower().map({
    "germany": "Germany", "de": "Germany", "deu": "Germany", "german": "Germany",
    "japan": "Japan", "jp": "Japan", "jpn": "Japan", "japanese": "Japan"
})
raw["Year"] = pd.to_numeric(raw["Year"], errors="coerce")
raw["Leadership_context_human"] = raw["Leadership_context_human"].astype("string").str.strip().str.upper()
raw["Information_valence_human"] = raw["Information_valence_human"].astype("string").str.strip().str.lower()
raw["_text_type"] = (raw["Text_type"].astype("string").str.strip().str.lower()
                     .str.replace(r"[\s-]+", "_", regex=True))
raw["_nonempty_text"] = raw["Text"].notna() & raw["Text"].astype("string").str.strip().ne("").fillna(False)
print("Original rows:", len(raw), "| Columns:", len(raw.columns))
display(raw.groupby(["_file_name", "Text_type"], dropna=False).size()
        .rename("n_rows").reset_index().sort_values(["_file_name", "n_rows"], ascending=[True, False]))

## 3. Eligibility, segmentation, and data-quality audit

The **analytical sample**, not all extracted rows, must underlie every descriptive figure and every future sentiment result. The following checks retain only (i) leader-authored eligible text types, (ii) the non-duplicated level of hierarchical segmentation, (iii) non-empty texts, (iv) properly coded Context and Valence, and (v) valid 2020–2025 company/country metadata. Exclusions are counted **sequentially**, so the final number is reproducible. Coding ambiguity is *flagged*, not silently removed; it may have been adjudicated.

**Segmentation safeguard:** A source ID such as `DEU20A01-00` is omitted only when a `DEU20A01-01` or similar eligible, non-empty child exists in the same company. A source without eligible children remains eligible. Check this logic against your final dataset before estimating sentiment.

In [ ]:
id_parts = raw["ID"].str.extract(r"^(?P<base>.+)-(?P<suffix>\d{2})$")
raw["_id_base"] = id_parts["base"]
raw["_id_suffix"] = id_parts["suffix"]
child_rows = raw.loc[raw["_id_suffix"].notna() & raw["_id_suffix"].ne("00")
                     & raw["_text_type"].isin(ANALYTICAL_TEXT_TYPES) & raw["_nonempty_text"]]
child_pairs = set(zip(child_rows["Company"], child_rows["_id_base"]))
raw["_superseded_original"] = [
    suffix == "00" and (company, base) in child_pairs
    for company, base, suffix in zip(raw["Company"], raw["_id_base"], raw["_id_suffix"])
]

checks = {
    "excluded_text_type": ~raw["_text_type"].isin(ANALYTICAL_TEXT_TYPES),
    "superseded_original_block": raw["_superseded_original"],
    "empty_text": ~raw["_nonempty_text"],
    "invalid_context_or_valence": (
        ~raw["Leadership_context_human"].isin(CONTEXT_ORDER) |
        ~raw["Information_valence_human"].isin(VALENCE_ORDER)
    ),
    "invalid_company_country_year": (
        raw["Company"].ne(raw["_file_company"]).fillna(True) |
        raw["Country"].ne(raw["Company"].map(COMPANY_COUNTRY)).fillna(True) |
        ~raw["Year"].isin(YEARS)
    )
}
remaining = pd.Series(True, index=raw.index)
attrition_rows = [{"step": "Raw observations", "excluded_at_step": 0, "remaining": int(remaining.sum())}]
for reason, mask in checks.items():
    n_excluded = int((remaining & mask.fillna(True)).sum())
    remaining &= ~mask.fillna(True)
    attrition_rows.append({"step": reason, "excluded_at_step": n_excluded,
                           "remaining": int(remaining.sum())})

analysis = raw.loc[remaining].copy().reset_index(drop=True)
if analysis.empty:
    raise ValueError("There are no eligible coded leadership segments. Check Text_type and labels.")
if analysis["ID"].isna().any() or analysis["ID"].eq("").fillna(True).any():
    raise ValueError("The analytical sample has missing IDs. Restore unique IDs before analysis.")
if analysis["ID"].duplicated().any():
    examples = analysis.loc[analysis["ID"].duplicated(keep=False), "ID"].head(10).tolist()
    raise ValueError(f"Duplicate analytical IDs found: {examples}. Resolve them before continuing.")

# Text lengths are descriptive whitespace/word proxies, not transformer model tokens.
analysis["word_count"] = analysis["Text"].astype(str).str.findall(r"(?u)\b\w+(?:[’'-]\w+)*\b").str.len()
analysis["char_count"] = analysis["Text"].astype(str).str.len()
analysis["_ambiguous"] = (analysis["Coding_ambiguity"].astype("string").str.strip().str.lower()
                          .isin(["yes", "y", "true", "1", "ambiguous", "oui"]))

attrition = pd.DataFrame(attrition_rows)
save_table(attrition, "table_01_sample_attrition")
missingness = (raw[required_columns].isna().sum().rename("missing_n").to_frame()
               .assign(missing_pct=lambda x: (x["missing_n"] / len(raw) * 100).round(2))
               .reset_index(names="column"))
save_table(missingness, "audit_missingness_raw")
quality = pd.DataFrame([{
    "raw_rows": len(raw), "eligible_segments": len(analysis),
    "duplicate_ids_raw": int(raw["ID"].dropna().duplicated().sum()),
    "duplicate_texts_analytical": int(analysis["Text"].duplicated().sum()),
    "ambiguous_segments_analytical": int(analysis["_ambiguous"].sum()),
    "superseded_original_blocks": int(raw["_superseded_original"].sum()),
    "invalid_company_file_rows": int(raw["Company"].ne(raw["_file_company"]).fillna(True).sum()),
    "unexpected_country_rows": int(raw["Country"].ne(raw["Company"].map(COMPANY_COUNTRY)).fillna(True).sum()),
}])
quality.to_csv(QUALITY_DIR / "dataset_audit_summary.csv", index=False)
display(attrition)
display(quality)
print("Analytical sample:", len(analysis), "segments")

## 4. Corpus composition by country, company and year

Report **both** the number of segments and their word volume. A country with more segments need not contribute more words. The company–year grid reveals missing years and uneven exposure; a zero denotes no eligible observations and does not, by itself, establish that no source document existed. **Country and company are perfectly nested** in this four-company sample, so country patterns can also reflect company-specific practices.

In [ ]:
company_summary = (analysis.groupby(["Country", "Company"], observed=True)
                   .agg(n_segments=("ID", "size"), total_words=("word_count", "sum"),
                        median_words=("word_count", "median"), mean_words=("word_count", "mean"),
                        n_years=("Year", "nunique"))
                   .reset_index())
company_summary["share_of_segments_pct"] = (100 * company_summary["n_segments"] / len(analysis)).round(1)
company_summary["share_of_words_pct"] = (100 * company_summary["total_words"] / analysis["word_count"].sum()).round(1)
country_summary = (analysis.groupby("Country", observed=True)
                   .agg(n_segments=("ID", "size"), total_words=("word_count", "sum"),
                        median_words=("word_count", "median"), n_companies=("Company", "nunique"))
                   .reindex(COUNTRY_ORDER).reset_index())
company_year_counts = (pd.crosstab(analysis["Company"], analysis["Year"])
                       .reindex(index=COMPANY_ORDER, columns=YEARS, fill_value=0).astype(int))
save_table(company_summary, "table_02_corpus_by_company")
save_table(country_summary, "table_03_corpus_by_country")
save_table(company_year_counts, "table_04_company_year_segments", index=True)
display(company_summary.round(1))
display(country_summary.round(1))
display(company_year_counts)

fig, ax = plt.subplots(figsize=(8.5, 3.6))
plot_data = company_year_counts.to_numpy()
im = ax.imshow(plot_data, aspect="auto", cmap="Blues")
ax.set_xticks(range(len(YEARS)), YEARS)
ax.set_yticks(range(len(COMPANY_ORDER)), COMPANY_ORDER)
ax.set_xlabel("Reporting year")
ax.set_title("Analytical leadership segments by company and year")
for i in range(plot_data.shape[0]):
    for j in range(plot_data.shape[1]):
        v = plot_data[i, j]
        ax.text(j, i, str(v), ha="center", va="center",
                color="white" if v > plot_data.max() / 2 else "black", fontsize=9)
fig.colorbar(im, ax=ax, label="Number of segments", shrink=0.85)
save_figure(fig, "fig_01_company_year_coverage", "Eligible coded leadership segments by company and reporting year.")

## 5. Leadership Context distributions

Context frequency addresses *what is discussed*, not the sentiment of the language. Compare overall shares and within-country/company percentages. The company chart is essential: any Germany–Japan contrast could be driven by one firm. `OTHER` is retained but should be reviewed if it occupies a large share.

In [ ]:
def categorical_crosstab(index, col, rows, cols, prefix):
    counts = pd.crosstab(analysis[index], analysis[col]).reindex(index=rows, columns=cols, fill_value=0)
    shares = counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0).mul(100)
    save_table(counts, f"{prefix}_counts", index=True)
    save_table(shares.round(2), f"{prefix}_row_pct", index=True)
    return counts, shares

context_country_n, context_country_pct = categorical_crosstab(
    "Country", "Leadership_context_human", COUNTRY_ORDER, CONTEXT_ORDER, "table_05_context_by_country")
context_company_n, context_company_pct = categorical_crosstab(
    "Company", "Leadership_context_human", COMPANY_ORDER, CONTEXT_ORDER, "table_06_context_by_company")
context_overall = (analysis["Leadership_context_human"].value_counts()
                   .reindex(CONTEXT_ORDER, fill_value=0).rename_axis("Leadership_context")
                   .reset_index(name="n_segments"))
context_overall["share_pct"] = 100 * context_overall["n_segments"] / len(analysis)
save_table(context_overall.round(2), "table_context_overall")
display(context_country_n)
display(context_country_pct.round(1))

def plot_stacked_share(shares, title, name, caption):
    fig, ax = plt.subplots(figsize=(9.5, max(2.8, len(shares) * 0.65 + 1.8)))
    shares.plot(kind="barh", stacked=True, ax=ax, width=0.72, colormap="tab20c")
    ax.set_xlim(0, 100)
    ax.set_xlabel("Share of eligible segments (%)")
    ax.set_ylabel("")
    ax.set_title(title)
    ax.invert_yaxis()
    ax.legend(title="Leadership Context", bbox_to_anchor=(1.01, 1), loc="upper left")
    ax.grid(axis="x", alpha=0.2)
    ax.set_axisbelow(True)
    save_figure(fig, name, caption)

plot_stacked_share(context_country_pct, "Leadership Context composition by country",
                   "fig_02_context_by_country", "Within-country shares of seven Leadership Context categories.")
plot_stacked_share(context_company_pct, "Leadership Context composition by company",
                   "fig_03_context_by_company", "Within-company shares of seven Leadership Context categories.")

## 6. Information Valence distributions

Information Valence is the **manual coding of underlying informational favorability**, not linguistic sentiment. Examine overall and within-group shares; differences here are important background for the later models because Valence is an explicit control variable.

In [ ]:
valence_country_n, valence_country_pct = categorical_crosstab(
    "Country", "Information_valence_human", COUNTRY_ORDER, VALENCE_ORDER, "table_07_valence_by_country")
valence_company_n, valence_company_pct = categorical_crosstab(
    "Company", "Information_valence_human", COMPANY_ORDER, VALENCE_ORDER, "table_08_valence_by_company")
display(valence_country_n)
display(valence_country_pct.round(1))

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, shares, title in zip(axes, [valence_country_pct, valence_company_pct],
                             ["Country", "Company"]):
    shares.plot.bar(stacked=True, ax=ax, width=0.72, colormap="Pastel1", legend=False)
    ax.set_ylim(0, 100)
    ax.set_title(f"Information Valence by {title.lower()}")
    ax.set_xlabel("")
    ax.set_ylabel("Within-group share (%)")
    ax.tick_params(axis="x", rotation=35)
    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title="Information Valence", loc="upper center", bbox_to_anchor=(0.5, 1.10), ncol=3)
fig.tight_layout()
save_figure(fig, "fig_04_valence_by_country_company",
            "Within-group manual Information Valence distributions by country and company.")

## 7. Joint distribution: Context × Information Valence

This cross-tabulation is important for your design: it documents whether the information coded in each context is predominantly favorable, unfavorable or neutral. It is **not** a sentiment result. Inspect country-specific distributions as well, especially the sparsity of Context × Country × Valence combinations that will appear in the subsequent models.

In [ ]:
context_valence_n = (pd.crosstab(analysis["Leadership_context_human"], analysis["Information_valence_human"])
                     .reindex(index=CONTEXT_ORDER, columns=VALENCE_ORDER, fill_value=0))
context_valence_pct = context_valence_n.div(context_valence_n.sum(axis=1).replace(0, np.nan), axis=0).mul(100)
save_table(context_valence_n, "table_09_context_valence_counts", index=True)
save_table(context_valence_pct.round(2), "table_09_context_valence_row_pct", index=True)
display(context_valence_n)
display(context_valence_pct.round(1))

fig, ax = plt.subplots(figsize=(7.5, 5.0))
image = ax.imshow(context_valence_pct.to_numpy(dtype=float), aspect="auto", vmin=0, vmax=100, cmap="Blues")
ax.set_xticks(range(len(VALENCE_ORDER)), VALENCE_ORDER)
ax.set_yticks(range(len(CONTEXT_ORDER)), CONTEXT_ORDER)
ax.set_title("Information Valence within Leadership Context")
ax.set_xlabel("Manually coded Information Valence")
for i in range(len(CONTEXT_ORDER)):
    for j in range(len(VALENCE_ORDER)):
        v = context_valence_pct.iat[i, j]
        ax.text(j, i, "n/a" if pd.isna(v) else f"{v:.1f}%", ha="center", va="center",
                color="white" if pd.notna(v) and v > 55 else "black")
fig.colorbar(image, ax=ax, label="Within-context share (%)", shrink=0.9)
save_figure(fig, "fig_05_valence_within_context",
            "Distribution of manually coded Information Valence within each Leadership Context.")

support_grid = pd.MultiIndex.from_product([COUNTRY_ORDER, CONTEXT_ORDER, VALENCE_ORDER],
                                          names=["Country", "Leadership_context", "Information_valence"])
support = (analysis.groupby(["Country", "Leadership_context_human", "Information_valence_human"], observed=True)
           .size().rename_axis(["Country", "Leadership_context", "Information_valence"])
           .reindex(support_grid, fill_value=0).reset_index(name="n_segments"))
save_table(support, "table_10_country_context_valence_support")
support["within_country_context_pct"] = (
    support["n_segments"].div(support.groupby(["Country", "Leadership_context"])["n_segments"]
                              .transform("sum").replace(0, np.nan)).mul(100)
)
save_table(support.round(2), "table_10_country_context_valence_within_group_pct")
print("Context × Country × Valence cells with fewer than five segments:",
      f"{(support['n_segments'] < 5).sum()} of {len(support)}")
print("NOTE: These are segment counts, not independent company/document observations.")
display(support.pivot_table(index=["Country", "Leadership_context"],
                            columns="Information_valence", values="n_segments", fill_value=0).astype(int))

## 8. Segment length, possible long-text cases, and weighting

The analysis unit is a segment/paragraph, but lengths can vary. Display the **median and interquartile range**, not only the mean. The boxplots exclude outlier markers for legibility; the top-length table retains those observations for auditing. `word_count` is a rough proxy and **cannot** establish whether a text exceeds the models' 510 content-token chunk size. Exact token limits must later be checked using each model's tokenizer. Longer segments contribute more language; segment-weighted and word-weighted descriptive statistics answer different questions.

In [ ]:
length_summary = (analysis.groupby(["Country", "Company"], observed=True)["word_count"]
                  .agg(n="count", mean="mean", median="median", std="std", minimum="min",
                       q25=lambda s: s.quantile(0.25), q75=lambda s: s.quantile(0.75),
                       p95=lambda s: s.quantile(0.95), maximum="max").reset_index())
save_table(length_summary.round(2), "table_11_text_length_by_company")
longest = analysis.nlargest(min(25, len(analysis)), "word_count")[["ID", "Company", "Country", "Year", "Text_type", "word_count", "char_count"]]
save_table(longest, "audit_25_longest_segments")
display(length_summary.round(1))
print("Segments longer than 350 words (screening only, NOT exact model token count):",
      int((analysis["word_count"] > 350).sum()))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))
vals = [analysis.loc[analysis["Company"].eq(c), "word_count"].values for c in COMPANY_ORDER]
axes[0].boxplot(vals, tick_labels=COMPANY_ORDER, showfliers=False)
axes[0].set_ylabel("Words per segment")
axes[0].set_title("Segment length by company (outliers hidden)")
axes[0].grid(axis="y", alpha=0.2)
axes[1].hist(analysis["word_count"], bins=min(35, max(5, int(np.sqrt(len(analysis))))), edgecolor="white")
axes[1].set_xlabel("Words per segment")
axes[1].set_ylabel("Number of segments")
axes[1].set_title("Full segment-length distribution")
axes[1].grid(axis="y", alpha=0.2)
fig.tight_layout()
save_figure(fig, "fig_06_segment_lengths", "Segment-level word counts by company and across the full analytical corpus.")

## 9. Reporting-format and communicator composition

Before controlling for `Communication_type` or considering `Source_type` and `Communicator_role`, inspect whether each appears in both countries. A characteristic that occurs only in one country or company cannot be cleanly separated from national environment. The thesis already notes that a `Covid` flag defined entirely by reporting year should **not** be entered together with a complete set of year indicators.

In [ ]:
metadata_columns = ["Text_type", "Communication_type", "Source_type", "Communicator_role", "Covid"]
metadata_summaries = {}
for column in metadata_columns:
    temp = (analysis.groupby(["Country", "Company", column], dropna=False, observed=True)
            .size().rename("n_segments").reset_index())
    metadata_summaries[column] = temp
    save_table(temp, f"table_metadata_{column.lower()}")
    print(f"\n{column}: {analysis[column].nunique(dropna=True)} observed categories")
    display(pd.crosstab(analysis["Country"], analysis[column], dropna=False))

# A direct year-COVID diagnostic: do not infer what individual code values mean.
if "Covid" in analysis:
    covid_by_year = pd.crosstab(analysis["Year"], analysis["Covid"], dropna=False)
    save_table(covid_by_year, "audit_covid_by_year", index=True)
    print("COVID indicator by year:")
    display(covid_by_year)

## 10. Ambiguous classifications and other review targets

The manual–AI agreement exercise is already documented in the thesis. This is **not** a repeat of Cohen's κ: it summarizes any remaining `Coding_ambiguity` flags in the *final* analytical corpus. A flagged row can remain in the analysis if the final classification has already been adjudicated. Review your original audit trail before any further exclusions.

In [ ]:
ambiguity_summary = (analysis.groupby(["Country", "Company"], observed=True)
                     .agg(n_segments=("ID", "size"), ambiguous_n=("_ambiguous", "sum"),
                          other_context_n=("Leadership_context_human", lambda s: s.eq("OTHER").sum()))
                     .reset_index())
ambiguity_summary["ambiguous_pct"] = 100 * ambiguity_summary["ambiguous_n"] / ambiguity_summary["n_segments"]
ambiguity_summary["other_pct"] = 100 * ambiguity_summary["other_context_n"] / ambiguity_summary["n_segments"]
save_table(ambiguity_summary.round(2), "table_12_ambiguity_and_other")
ambiguity_details = analysis.loc[analysis["_ambiguous"],
     ["ID", "Company", "Year", "Leadership_context_human", "Information_valence_human",
      "Alternative_context", "Comment"]]
save_table(ambiguity_details, "audit_ambiguous_segments")
display(ambiguity_summary.round(1))
print("Ambiguous rows exported for manual review:", len(ambiguity_details))

## 11. Country-comparison sensitivity to unequal sample composition

Mizuno has no 2020 English statement in the thesis sample. Compare (a) the entire 2020–2025 pooled segments, (b) the shared 2021–2025 period, and (c) a **company-balanced** estimate where each of the two firms in a country receives equal descriptive weight. These are *diagnostics of corpus composition*, not formal sensitivity tests of sentiment or corrections for four-company confounding. Large differences mean the eventual country comparisons are sensitive to sample balance.

In [ ]:
def country_shares_for_period(data, var, categories):
    counts = pd.crosstab(data["Country"], data[var]).reindex(
        index=COUNTRY_ORDER, columns=categories, fill_value=0)
    return counts.div(counts.sum(axis=1).replace(0, np.nan), axis=0).mul(100)

comparison_rows = []
for variable, categories, company_pct, label in [
    ("Leadership_context_human", CONTEXT_ORDER, context_company_pct, "Leadership Context"),
    ("Information_valence_human", VALENCE_ORDER, valence_company_pct, "Information Valence")
]:
    pooled = country_shares_for_period(analysis, variable, categories)
    shared_period = country_shares_for_period(analysis.loc[analysis["Year"].ge(2021)], variable, categories)
    for country in COUNTRY_ORDER:
        companies = [company for company in COMPANY_ORDER if COMPANY_COUNTRY[company] == country]
        balanced = company_pct.reindex(companies).mean(axis=0)
        for category in categories:
            comparison_rows.append({
                "dimension": label, "Country": country, "category": category,
                "pooled_2020_2025_pct": pooled.loc[country, category],
                "pooled_2021_2025_pct": shared_period.loc[country, category],
                "equal_company_weight_pct": balanced.loc[category],
                "change_shared_years_pp": shared_period.loc[country, category] - pooled.loc[country, category],
                "change_company_balanced_pp": balanced.loc[category] - pooled.loc[country, category]
            })
comparability = pd.DataFrame(comparison_rows)
save_table(comparability.round(2), "table_14_country_composition_sensitivity")
display(comparability.round(1))
print("Differences are in percentage points; they do not measure sentiment or causal country effects.")

## 12. Optional lexical description (not the sentiment input)

The top terms can help describe the subject matter of the corpus and spot extraction artefacts. English stopwords are removed **for this lexical chart only**; do *not* remove stopwords or reprocess the original segment text before sending it to FinBERT / Financial-RoBERTa. Counts are normalized to **10,000 retained (non-stopword) tokens** to make different-sized company corpora more comparable. Interpret frequent tokens as word usage, not leadership contexts or sentiment.

In [ ]:
term_rows = []
for company in COMPANY_ORDER:
    texts = analysis.loc[analysis["Company"].eq(company), "Text"].astype(str).tolist()
    if not texts:
        continue
    vectorizer = CountVectorizer(lowercase=True, stop_words="english",
                                 token_pattern=r"(?u)\b[a-zA-Z]{3,}\b", ngram_range=(1, 1))
    try:
        X = vectorizer.fit_transform(texts)
    except ValueError:  # all words filtered out
        warnings.warn(f"No lexical vocabulary for {company}.")
        continue
    vocab = vectorizer.get_feature_names_out()
    counts = np.asarray(X.sum(axis=0)).ravel()
    total_retained_tokens = counts.sum()
    if total_retained_tokens == 0:
        continue
    for idx in np.argsort(-counts)[:20]:
        term_rows.append({"Company": company, "word": str(vocab[idx]),
                          "count": int(counts[idx]),
                          "per_10000_retained_tokens": float(counts[idx] * 10000 / total_retained_tokens)})
term_frequency = pd.DataFrame(term_rows, columns=["Company", "word", "count", "per_10000_retained_tokens"])
save_table(term_frequency.round(2), "table_13_top20_words_by_company")

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for company, ax in zip(COMPANY_ORDER, axes.flat):
    sub = term_frequency.loc[term_frequency["Company"].eq(company)].head(12).iloc[::-1]
    if len(sub):
        ax.barh(sub["word"], sub["per_10000_retained_tokens"])
    else:
        ax.text(0.5, 0.5, "No vocabulary", transform=ax.transAxes, ha="center")
    ax.set_title(company)
    ax.set_xlabel("Per 10,000 retained tokens")
    ax.grid(axis="x", alpha=0.2)
    ax.set_axisbelow(True)
fig.suptitle("Most frequent non-stopword terms (descriptive only)", y=1.01)
fig.tight_layout()
save_figure(fig, "fig_07_top_terms_by_company",
            "Most frequent words by company, normalized per 10,000 tokens after stopword filtering (descriptive only).")

## 13. Final outputs and implications for the sentiment stage

Use the summary tables and figures to write a **Dataset description / Preliminary descriptive analysis** subsection, reporting at minimum: the final number of segments, country/company/year coverage, Context and Valence composition, segment-length distribution, and whether important Context × Country × Valence cells are sparse. For the main text, **Figure 1 (coverage), Figures 2–3 (Context), Figure 4 (Valence), and Figure 5 (joint distributions)** are usually the strongest candidates; lexical charts and full metadata cross-tabs can move to an appendix.

**Before testing H1–H3:**

- Run both sentiment models on the exact same `analysis` IDs and original `Text`, without changing segmentation or dropping classes after seeing scores.
- Validate sentiment predictions against a separate human *linguistic sentiment* subsample; this is different from Information Valence validation.
- Keep both model outcomes separately (`P(positive) - P(negative)`), and report distributions, inter-model concordance, and extreme disagreements before fitting regressions.
- Retain a trustworthy **source-document identifier** for clustering/document-level sensitivity checks. The current workbook schema does not establish a unique document ID; company-year should **not** be assumed to identify a document.
- Because there are only **four companies (two per country)**, country is confounded with company. Segment-level nominal p-values may greatly overstate independent information, and four company clusters cannot support reliable conventional cluster-robust inference. Interpret H2/H3 as descriptive within-sample associations, not causal country/cultural effects.
- Do not include a year-defined `Covid` indicator alongside a complete set of year dummy variables.

**No hypothesis tests are performed in this notebook.** Those belong in the post-sentiment notebook where the dependent variable actually exists. The outputs below are reproducible files, not invented numerical findings.

In [ ]:
figure_manifest = pd.DataFrame(figure_records)
save_table(figure_manifest, "figure_manifest")
print("Generated", len(figure_records), "figures (PNG + PDF each)")
print("Figure paths:")
display(figure_manifest)
print("CSV tables:", len(list(TABLES_DIR.glob("*.csv"))))
print("Data not modified. Analytical IDs are stored in the analysis dataframe.")

# Optional reproducibility anchor: table of analysis IDs and derived lengths, without exporting texts.
save_table(analysis[["ID", "Company", "Country", "Year", "Text_type",
                     "Leadership_context_human", "Information_valence_human",
                     "word_count", "char_count"]], "analytical_sample_manifest")